# Qwen3-VL-32B-Instruct 적용 버전

베이스라인(Qwen2.5-VL-3B)에서 Qwen3-VL-32B로 교체한 버전입니다.

**변경 사항 요약:**
- 모델: `Qwen2.5-VL-3B` → `Qwen3-VL-32B-Instruct`
- 이미지 해상도: 384px 고정 → 동적 해상도 (256~1280px)
- 학습 데이터: 200개 샘플 → 전체 데이터
- LoRA rank: 8 → 16
- 학습률: 1e-4 → 2e-5
- Epoch: 1 → 3
- compute_dtype: float16 → bfloat16 (A100 최적화)
- 데이터: GitHub 레포에서 클론

A100 GPU (40GB) 환경 / Colab Pro에서 실행하세요.

# 환경 준비

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [1]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" "datasets>=3.0.0" "qwen-vl-utils"
!pip -q install "pandas==2.2.2" "pillow<12"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 71.0 MB/s eta 0:00:00


In [ ]:
import torch
import pandas
import PIL
import transformers

print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Pandas version:", pandas.__version__)
print("Pillow version:", PIL.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

Torch version: 2.10.0+cu128
Transformers version: 5.5.0.dev0
Pandas version: 2.2.2
Pillow version: 11.3.0
CUDA version: 12.8
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [ ]:
# 압축 해제
!unzip "/content/2026_15_2_ai_DataSet.zip" -d "/content/"


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

# 라이브러리, 데이터, 설정

In [ ]:
!ls /content/
!find /content -name "train.csv"
!find /content -name "test.csv"

2026_15_2_ai_DataSet.zip  sample_data		 test.csv
dev			  sample_submission.csv  train
dev.csv			  test			 train.csv
/content/train.csv
/content/test.csv


In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    get_peft_model_state_dict,
    set_peft_model_state_dict
)
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_ID   = "Qwen/Qwen3-VL-32B-Instruct"

# 작은 글씨/OCR 문제면 나중에 512*512 ~ 1536*1536도 시도 가능
MIN_PIXELS = 256 * 256
MAX_PIXELS = 1280 * 1280

MAX_NEW_TOKENS = 2
EPOCHS = 3
SEED   = 42

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"전체 학습 데이터: {len(train_df)}개")
print(f"테스트 데이터:    {len(test_df)}개")
print(train_df.head(2))

Device: cuda


# 모델, Processor

32B 모델 다운로드: 약 60~70GB, 20~40분 소요됩니다.

In [ ]:
# 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Processor
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    trust_remote_code=True,
)

# base model
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 선택지 토큰 확인
for ch in ["a", "b", "c", "d"]:
    ids = processor.tokenizer.encode(ch, add_special_tokens=False)
    print(ch, ids)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1058 [00:00<?, ?it/s]

trainable params: 39,845,888 || all params: 33,397,235,952 || trainable%: 0.1193


# 프롬프트 템플릿

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert visual multiple-choice assistant for recycling images. "
    "Use the visual evidence in the image, compare all options carefully, "
    "and answer with exactly one lowercase letter among a, b, c, d. "
    "Do not output any explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n\n"
        f"Options:\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "Use only the visible evidence in the image. "
        "Compare all options carefully. "
        "Return only one lowercase letter: a, b, c, or d."
    )

# Custom Dataset, Collator

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        prompt_messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text}
            ]}
        ]

        answer = str(row["answer"]).strip().lower()

        return {
            "prompt_messages": prompt_messages,
            "image": img,
            "answer": answer
        }


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        assert len(batch) == 1, "현재 코드는 batch_size=1 전용"

        sample = batch[0]
        img = sample["image"]
        prompt_messages = sample["prompt_messages"]

        if self.train:
            full_messages = prompt_messages + [
                {"role": "assistant", "content": [{"type": "text", "text": sample["answer"]}]}
            ]

            prompt_text = self.processor.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=False
            )
            full_text = self.processor.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False
            )

            enc = self.processor(
                text=[full_text],
                images=[img],
                padding=True,
                return_tensors="pt"
            )

            prompt_enc = self.processor(
                text=[prompt_text],
                images=[img],
                padding=True,
                return_tensors="pt"
            )

            labels = enc["input_ids"].clone()

            # prompt 구간은 loss 제외
            prompt_len = int(prompt_enc["attention_mask"][0].sum().item())
            labels[:, :prompt_len] = -100

            # pad도 loss 제외
            pad_id = self.processor.tokenizer.pad_token_id
            if pad_id is not None:
                labels[labels == pad_id] = -100

            enc["labels"] = labels
            return enc

        else:
            prompt_text = self.processor.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )

            enc = self.processor(
                text=[prompt_text],
                images=[img],
                padding=True,
                return_tensors="pt"
            )
            return enc

# DataLoader

In [ ]:
# train/valid 셔플 분할
train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

split = int(len(train_df) * 0.9)
train_subset = train_df.iloc[:split].reset_index(drop=True)
valid_subset = train_df.iloc[split:].reset_index(drop=True)

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(processor, train=True),
    num_workers=0
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(processor, train=True),
    num_workers=0
)

print(f"Train samples: {len(train_subset)}")
print(f"Valid samples: {len(valid_subset)}")
print(f"Train batches: {len(train_loader)}, Valid batches: {len(valid_loader)}")

Train batches: 180, Valid batches: 20


# Fine-tuning

In [ ]:
from tqdm.auto import tqdm
from torch.nn.utils import clip_grad_norm_

model_device = next(model.parameters()).device
GRAD_ACCUM = 8

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    int(num_training_steps * 0.03),
    num_training_steps
)

model.config.use_cache = False
optimizer.zero_grad(set_to_none=True)

# a/b/c/d 토큰 ID
choice_token_ids = {}
for ch in ["a", "b", "c", "d"]:
    ids = processor.tokenizer.encode(ch, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"'{ch}'가 1토큰이 아닙니다: {ids}")
    choice_token_ids[ch] = ids[0]

def predict_choice_by_logits(row):
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(
        str(row["question"]),
        str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
    )

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text}
        ]}
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model_device)

    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**inputs)
            next_token_logits = outputs.logits[:, -1, :]

    scores = {
        ch: next_token_logits[0, tok_id].item()
        for ch, tok_id in choice_token_ids.items()
    }
    pred = max(scores, key=scores.get)
    return pred, scores


@torch.no_grad()
def evaluate_accuracy(eval_df, max_samples=None):
    model.eval()

    if max_samples is None:
        total = len(eval_df)
    else:
        total = min(len(eval_df), max_samples)

    correct = 0

    for i in tqdm(range(total), desc="Valid Acc", unit="sample"):
        row = eval_df.iloc[i]
        pred, _ = predict_choice_by_logits(row)
        gold = str(row["answer"]).strip().lower()
        correct += int(pred == gold)

    return correct / total


best_acc = -1.0
best_peft_state = None
SAVE_DIR_BEST = "/content/qwen3_vl_32b_lora_best"
SAVE_DIR_FINAL = "/content/qwen3_vl_32b_lora_final"

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(model_device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
            running = 0.0

    # valid loss
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad():
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid loss]", unit="batch"):
            vb = {k: v.to(model_device) for k, v in vb.items()}
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                out = model(**vb)
            val_loss += out.loss.item()
            val_steps += 1

    avg_val_loss = val_loss / max(val_steps, 1)
    val_acc = evaluate_accuracy(valid_subset, max_samples=min(200, len(valid_subset)))

    print(f"[Epoch {epoch+1}] valid loss: {avg_val_loss:.4f}")
    print(f"[Epoch {epoch+1}] valid acc : {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_peft_state = {
            k: v.detach().cpu().clone()
            for k, v in get_peft_model_state_dict(model).items()
        }
        model.save_pretrained(SAVE_DIR_BEST)
        processor.save_pretrained(SAVE_DIR_BEST)
        print(f"Best model saved to: {SAVE_DIR_BEST}")

# best adapter 다시 로드
if best_peft_state is not None:
    set_peft_model_state_dict(model, best_peft_state)
    print(f"Best adapter reloaded in memory. best_acc={best_acc:.4f}")

# final 저장
model.save_pretrained(SAVE_DIR_FINAL)
processor.save_pretrained(SAVE_DIR_FINAL)
print("Final saved:", SAVE_DIR_FINAL)

/tmp/ipykernel_26503/761726610.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=False)


Epoch 1 [train]:   0%|          | 0/180 [00:00<?, ?batch/s]

/tmp/ipykernel_26503/761726610.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/tmp/ipykernel_26503/761726610.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):


Epoch 1 [valid]:   0%|          | 0/20 [00:00<?, ?batch/s]

[Epoch 1] valid loss: 16.2241


...

Epoch 2 [valid]:   0%|          | 0/20 [00:00<?, ?batch/s]

[Epoch 2] valid loss: 13.0257


Epoch 3 [train]:   0%|          | 0/180 [00:00<?, ?batch/s]

Epoch 3 [valid]:   0%|          | 0/20 [00:00<?, ?batch/s]

[Epoch 3] valid loss: 11.2733
Saved: /content/qwen3_vl_32b_lora


In [ ]:
!zip -r /content/qwen3_vl_32b_lora_adapter.zip /content/qwen3_vl_32b_lora_adapter

In [ ]:
from google.colab import files
files.download("/content/qwen3_vl_32b_lora_adapter.zip")

# Inference

In [ ]:
# Inference (logits 기반, 검증 방식과 동일하게 통일)
model.eval()

# a/b/c/d 토큰 ID 확인
choice_token_ids = {}
for ch in ["a", "b", "c", "d"]:
    ids = processor.tokenizer.encode(ch, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"'{ch}'가 1토큰이 아닙니다: {ids}")
    choice_token_ids[ch] = ids[0]

@torch.no_grad()
def predict_choice_by_logits(row):
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(
        str(row["question"]),
        str(row["a"]),
        str(row["b"]),
        str(row["c"]),
        str(row["d"])
    )

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text}
        ]}
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        outputs = model(**inputs)
        next_token_logits = outputs.logits[:, -1, :]

    scores = {
        ch: next_token_logits[0, tok_id].item()
        for ch, tok_id in choice_token_ids.items()
    }
    pred = max(scores, key=scores.get)
    return pred, scores

preds = []
all_scores = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    pred, scores = predict_choice_by_logits(row)
    preds.append(pred)
    all_scores.append(scores)

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds
})
submission.to_csv("/content/submission.csv", index=False)

print("Saved /content/submission.csv")
print(submission.head())

# 마지막 샘플 점수 확인
print("Last prediction:", preds[-1])
print("Last scores:", all_scores[-1])

Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

In [ ]:
model.eval()

preds = []
score_rows = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    pred, scores = predict_choice_by_logits(row)

    preds.append(pred)
    score_rows.append({
        "id": row["id"],
        "pred": pred,
        "score_a": scores["a"],
        "score_b": scores["b"],
        "score_c": scores["c"],
        "score_d": scores["d"],
    })

submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds
})
submission.to_csv("/content/submission.csv", index=False)

score_df = pd.DataFrame(score_rows)
score_df.to_csv("/content/inference_scores.csv", index=False)

print("Saved /content/submission.csv")
print("Saved /content/inference_scores.csv")
print(submission.head())

In [ ]:
!zip -j /content/inference_results.zip /content/submission.csv /content/inference_scores.csv

In [ ]:
from google.colab import files
files.download("/content/inference_results.zip")

In [ ]:
# 마지막 추론 결과 확인
print(output_text)